<h1>Import dependencies<h1>

In [ ]:
import tensorflow as tf
import os 
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense,RandomCrop,RandomZoom,RandomFlip,RandomBrightness,Dropout,
    Layer,RandomRotation,Input,LeakyReLU,GlobalAveragePooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import callbacks
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.losses import Loss
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import Metric
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras.models import Sequential

import kagglehub
import matplotlib.pyplot as plt
from uuid import uuid4

<h1>Download Dataset<h1>

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = './data'
path = kagglehub.dataset_download("kaustubhb999/tomatoleaf")

print("Path to dataset files:", path)

<h1>Define Variables<h1>

In [ ]:
TRAIN_PATH = os.path.join(path,'tomato','train')
VAL_PATH = os.path.join(path,'tomato',"val")
W,H,D = 128,128,3
N_CLASSES = len(os.listdir(TRAIN_PATH))
LEARNING_RATE = 1e-3
EPOCHS = 40
BATCH_SIZE = 32
data_dict = {label:i for i,label in enumerate(sorted(os.listdir(TRAIN_PATH)))}
inverse_data_dict = {i:label for i,label in enumerate(sorted(os.listdir(TRAIN_PATH)))}
os.makedirs("./models", exist_ok=True)

In [ ]:
inverse_data_dict

<h1>Extract images paths and labels<h1>

In [ ]:
def get_data(dir_path):
    DATA = []
    TARGETS = []
    labels = sorted(os.listdir(dir_path))
    for label in labels:
        new_path = os.path.join(dir_path,label)
        images = os.listdir(new_path)

        for image in images:
            image_path = os.path.join(new_path,image)
            DATA.append(image_path)
            TARGETS.append(data_dict[label])


    return DATA,TARGETS

In [ ]:
TRAIN_DATA,TRAIN_TARGETS = get_data(TRAIN_PATH)
VAL_DATA,VAL_TARGETS = get_data(VAL_PATH)
SHUFFLE_BUFFER = len(TRAIN_DATA)

<h1>Creating Pipeline<h1>

<h5>Preprocessing<h5>

In [ ]:
def preprocessing(data,label):
    img = tf.io.decode_jpeg(tf.io.read_file(data),channels=D)
    img = tf.image.resize(img,size=[W,H])
    img = tf.cast(img,dtype=tf.float32)

    label = tf.cast(label,dtype=tf.int32)

    return img,label

In [ ]:
def get_pipeline(data,labels,shuffle_ok):
    DATASET = tf.data.Dataset.from_tensor_slices((data,labels))
    if shuffle_ok:
        DATASET = DATASET.shuffle(buffer_size=SHUFFLE_BUFFER)
    DATASET = DATASET.map(preprocessing,num_parallel_calls=tf.data.AUTOTUNE)
    DATASET = DATASET.cache()
    DATASET = DATASET.batch(BATCH_SIZE)
    DATASET = DATASET.prefetch(tf.data.AUTOTUNE)

    return DATASET

<h5>Data Partitioning<h5>

In [ ]:
DATA = get_pipeline(TRAIN_DATA,TRAIN_TARGETS,shuffle_ok=True) 

TEST = get_pipeline(VAL_DATA,VAL_TARGETS,shuffle_ok=False)  #For eval
TRAIN = DATA.take(int(len(DATA)*.9))                        #For train
VAL = DATA.skip(int(len(DATA)*.9))                          #For val

print(f"train: {len(TRAIN)}\ntest: {len(TEST)}\nval: {len(VAL)}")

In [ ]:
fig,ax = plt.subplots(4,3,figsize=(8,5))
ax =ax.flatten()
ds,lbl = TRAIN.as_numpy_iterator().next()
ds = ds/255.0

for i in range(12):
    ax[i].imshow(ds[i])
    ax[i].set_title(inverse_data_dict[int(lbl[i])])
    ax[i].axis('off')

fig.tight_layout()

plt.show()


<h1>Model Creating<h1>

In [ ]:
earlystop = callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True,start_from_epoch=10)

class callback_clz(Callback):
    def __init__(self):
        super(callback_clz,self).__init__()
        self.save_nb = 10
    def on_epoch_end(self,epoch,logs=None):
        if epoch%self.save_nb==0 and epoch!=0:
            model_name = f"./models/model_{epoch}.keras"
            model.save(model_name)
            print(f"\nmodel saved on {epoch}th epoch")

all_callbacks = [earlystop,callback_clz()]

In [ ]:
def aug_model():
    return Sequential([
        RandomRotation(0.03),
        RandomZoom(0.1),
        RandomBrightness(0.1),
        RandomFlip('horizontal')
    ])
    

In [ ]:
def base_model(input_shape=(W,H,D)):
    model = EfficientNetV2B0(input_shape=input_shape,weights='imagenet',include_top=False)
    model.trainable=False

    return model

In [ ]:
def Dense_layer(dense_units=[256,128,64],dropout_units=[0.3,0.2,0.1]):
    model = Sequential()

    for dense_unit,dropout_unit in zip(dense_units,dropout_units):
        model.add(Dense(units=dense_unit,activation="relu",kernel_initializer="he_normal"))
        model.add(Dropout(dropout_unit))

    return model


In [ ]:
def build_model(input_size):
    input_layer = Input(shape=input_size)

    x = aug_model()(input_layer)
    x = base_model()(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense_layer()(x)
    
    output_layer = Dense(units=N_CLASSES,activation="softmax")(x)

    model = Model(input_layer,output_layer)

    return model
    
    

In [ ]:
model = build_model(input_size=(W,H,D))
model.summary()

<h1>Compile and Train</h1>

In [ ]:
optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer,loss=SparseCategoricalCrossentropy(),metrics=['accuracy'])

In [ ]:

hist = model.fit(TRAIN,epochs=EPOCHS,validation_data=VAL,verbose=True,callbacks=all_callbacks) 
model.save('./models/model_latest.keras')

<h1>Evaluate</h1>

In [ ]:
loss,accuracy = model.evaluate(TEST) 
print(f"Loss: {loss}\nAccuracy: {accuracy}")

In [ ]:
fig = plt.figure()
plt.plot(hist.history['loss'], color='teal', label='loss')
plt.plot(hist.history['val_loss'], color='orange', label='val_loss')
fig.suptitle('Loss', fontsize=20)
plt.legend(loc="upper left")
plt.show()

fig = plt.figure()
plt.plot(hist.history['accuracy'], color='teal', label='accuracy')
plt.plot(hist.history['val_accuracy'], color='orange', label='val_accuracy')
fig.suptitle('Accuracy', fontsize=20)
plt.legend(loc="upper left")
plt.show()

In [ ]:
import numpy as np

<h1>Testing</h1>

In [ ]:
path = r'/kaggle/input/datasets/kaustubhb999/tomatoleaf/tomato/val/Tomato___Bacterial_spot/01d7f4fe-793f-4a9b-bc8b-8aa05200984f___GCREC_Bact.Sp 2984.JPG'
img = tf.io.decode_jpeg(tf.io.read_file(path),channels=D)
img = tf.image.resize(img,[H,W])
img = tf.cast(img,dtype=tf.float32)
pred = model.predict(tf.expand_dims(img,axis=0))
result = np.argmax(pred)
conf = pred[0][result]

plt.imshow(img/255)
plt.title(f"{inverse_data_dict[result]}\nconfidence {conf}")
plt.show()
